# Encapsulation in Python — Explained Like You Are 5

## Goal

**Encapsulation** means keeping an object's information and the methods that safely use that information together inside one class.

Imagine a medicine capsule:

- The useful ingredients are kept together inside it.
- The outside protects the ingredients from careless handling.
- You use the capsule in the intended way instead of touching every ingredient.

A class works similarly: it bundles data and behavior and provides safe ways to interact with them.

> **Big idea:** Keep related data and actions together, then control how the data is read or changed.

## Everyday example: an ATM

An ATM does not let you directly type a new bank balance. Instead, it gives you safe actions:

- `deposit(amount)`
- `withdraw(amount)`
- `get_balance()`

Those actions can check passwords, reject bad amounts, and prevent withdrawals larger than the balance.

This is encapsulation: the balance and the rules that protect it live together.

## Important words

| Word | Easy meaning |
|---|---|
| Encapsulation | Bundle data and its methods, then control access |
| Attribute | Information stored in an object |
| Public | Intended for normal use from anywhere |
| Protected convention | Intended for the class and its children |
| Private name | Name-mangled to discourage accidental direct access |
| Getter | Method or property that reads a value |
| Setter | Method or property that changes a value safely |
| Validation | Checking a value before accepting it |
| Invariant | A rule that must always remain true |

## Python access levels are naming conventions

Python does not use strict public/protected/private access modifiers like some languages. Instead, attribute names communicate how they should be used.

| Style | Example | Meaning |
|---|---|---|
| Public | `self.name` | Normal part of the class interface |
| Protected convention | `self._name` | Internal use or subclasses; outside code should be careful |
| Private/name-mangled | `self.__name` | Python changes the stored name to reduce accidental access |

> A leading underscore is a request, not a locked door. Double underscores use name mangling, not encryption or true security.

## 1. Public attributes — original example

Public attributes are directly readable and changeable. The `Person` below exposes `name` and `age`.

The separate `get_name()` function can read `person.name` because it is public.

In [1]:
class Person:
    def __init__(self, name, age):
        self.name = name  # Public attribute
        self.age = age    # Public attribute


def get_name(person):
    return person.name


person = Person("Krish", 34)
print(get_name(person))

Krish


### Inspecting the public object — original `dir()` idea

`dir(person)` lists the names available on the object. To keep the output readable, we filter out Python's many built-in names beginning with `__`.

In [2]:
visible_names = [name for name in dir(person) if not name.startswith("__")]
print(visible_names)
print(person.__dict__)

['age', 'name']
{'name': 'Krish', 'age': 34}


### The problem with completely unrestricted data

Public data is convenient, but outside code can accidentally create an impossible value.

The next line works even though a negative age does not make sense. This tells us that important data may need validation.

In [3]:
person.age = -500
print("Invalid age was accepted:", person.age)

Invalid age was accepted: -500


## 2. Protected attributes — original inheritance example

A name beginning with one underscore, such as `_name`, means:

> This is an internal detail. The class and its child classes may use it, but ordinary outside code should avoid depending on it.

Python still allows direct access. The underscore communicates programmer intent; it does not enforce a security rule.

In [4]:
class ProtectedPerson:
    def __init__(self, name, age, gender):
        self._name = name  # Protected convention
        self._age = age    # Protected convention
        self.gender = gender


class Employee(ProtectedPerson):
    def __init__(self, name, age, gender):
        super().__init__(name, age, gender)

    def introduce(self):
        return f"Employee name: {self._name}, age: {self._age}"


employee = Employee("Krish", 34, "Male")
print(employee._name)       # Possible, but outside code should be careful
print(employee.introduce()) # Preferred public method

Krish
Employee name: Krish, age: 34


## 3. Private names — original example

A name beginning with two underscores, such as `__name`, is **name-mangled** by Python.

Inside `Person`, the code can use `self.__name`. From outside, `person.__name` fails because Python stored it under a changed name.

We catch the expected error so the notebook continues running.

In [5]:
class PrivatePerson:
    def __init__(self, name, age, gender):
        self.__name = name  # Private/name-mangled attribute
        self.__age = age    # Private/name-mangled attribute
        self.gender = gender


private_person = PrivatePerson("Krish", 34, "Male")

try:
    print(private_person.__name)
except AttributeError as error:
    print(f"AttributeError: {error}")

AttributeError: 'PrivatePerson' object has no attribute '__name'


### What name mangling actually does — original `dir()` idea

Python changes `__name` to a name containing the class name, such as `_PrivatePerson__name`. This helps prevent accidental clashes and access.

It is still technically reachable, so double underscores are not a security system. Do not use the mangled name in ordinary application code.

In [6]:
private_names = [name for name in dir(private_person) if "name" in name or "age" in name]
print(private_names)
print(private_person.__dict__)

# Technically possible, but strongly discouraged in normal code:
print("Mangled access:", private_person._PrivatePerson__name)

['_PrivatePerson__age', '_PrivatePerson__name']
{'_PrivatePerson__name': 'Krish', '_PrivatePerson__age': 34, 'gender': 'Male'}
Mangled access: Krish


## 4. Getter and setter methods — original example

A **getter** reads private data. A **setter** changes it after checking whether the new value is allowed.

The setter below protects the rule that age must be positive.

In [7]:
class PersonWithMethods:
    def __init__(self, name, age):
        self.__name = name
        self.__age = age

    def get_name(self):
        return self.__name

    def set_name(self, name):
        self.__name = name

    def get_age(self):
        return self.__age

    def set_age(self, age):
        if age > 0:
            self.__age = age
        else:
            print("Age cannot be negative or zero.")


person_with_methods = PersonWithMethods("Krish", 34)

print(person_with_methods.get_name())
print(person_with_methods.get_age())

person_with_methods.set_age(35)
print(person_with_methods.get_age())

person_with_methods.set_age(-5)

Krish
34
35
Age cannot be negative or zero.


## 5. Properties: the modern Python style

Properties let us keep simple attribute syntax while running getter and setter logic behind the scenes.

```python
product.price          # Runs the getter
product.price = 450    # Runs the setter
```

The caller sees a normal-looking attribute. The class keeps control of validation.

In [8]:
class Product:
    def __init__(self, name, price):
        self.name = name
        self.price = price  # Uses the setter during setup

    @property
    def price(self):
        return self._price

    @price.setter
    def price(self, value):
        if not isinstance(value, (int, float)):
            raise TypeError("Price must be a number.")
        if value < 0:
            raise ValueError("Price cannot be negative.")
        self._price = value


product = Product("Python Book", 500)
print("Original price:", product.price)

product.price = 450
print("Updated price:", product.price)

try:
    product.price = -10
except ValueError as error:
    print(f"ValueError: {error}")

Original price: 500
Updated price: 450
ValueError: Price cannot be negative.


## 6. A read-only property

Sometimes outside code should read a value but change it only through safe methods.

The account below exposes `balance` as a property with no setter. Deposits and withdrawals are the approved ways to change it.

In [9]:
class BankAccount:
    def __init__(self, owner, opening_balance=0):
        if opening_balance < 0:
            raise ValueError("Opening balance cannot be negative.")
        self.owner = owner
        self.__balance = opening_balance

    @property
    def balance(self):
        return self.__balance

    def deposit(self, amount):
        self.__validate_positive_amount(amount)
        self.__balance += amount

    def withdraw(self, amount):
        self.__validate_positive_amount(amount)
        if amount > self.__balance:
            raise ValueError("Insufficient funds.")
        self.__balance -= amount

    def __validate_positive_amount(self, amount):
        if amount <= 0:
            raise ValueError("Amount must be positive.")


account = BankAccount("Krish", 1000)
account.deposit(500)
account.withdraw(300)
print("Balance:", account.balance)

try:
    account.balance = 1_000_000
except AttributeError as error:
    print(f"AttributeError: {error}")

Balance: 1200
AttributeError: property 'balance' of 'BankAccount' object has no setter


### Private helper methods

`__validate_positive_amount()` is an internal helper. Both deposit and withdrawal reuse it, so the validation rule is written once.

This is another part of encapsulation: internal steps stay inside the class while callers use the simple public methods.

## 7. Protecting a mutable collection

Lists are mutable. If a getter returns the real internal list, outside code can change it without following the class's rules.

The `students` property below returns an immutable tuple. Changes must go through `add_student()`, which validates names and prevents duplicates.

In [10]:
class Classroom:
    def __init__(self):
        self.__students = []

    @property
    def students(self):
        return tuple(self.__students)

    def add_student(self, name):
        clean_name = name.strip()
        if not clean_name:
            raise ValueError("Student name cannot be empty.")
        if clean_name in self.__students:
            raise ValueError("Student is already in the classroom.")
        self.__students.append(clean_name)


classroom = Classroom()
classroom.add_student("Aarav")
classroom.add_student("Riya")
print(classroom.students)

try:
    classroom.students.append("Kabir")
except AttributeError as error:
    print(f"AttributeError: {error}")

('Aarav', 'Riya')
AttributeError: 'tuple' object has no attribute 'append'


## 8. Encapsulation with inheritance

A child class should normally use the parent's public or protected interface instead of reaching into the parent's private names.

`PremiumAccount` uses the inherited public `deposit()` method to add a reward. It does not directly modify `__balance`.

In [11]:
class PremiumAccount(BankAccount):
    def add_reward(self, reward):
        self.deposit(reward)
        print(f"Reward of {reward} added safely.")


premium = PremiumAccount("Riya", 2000)
premium.add_reward(100)
print("Premium balance:", premium.balance)

try:
    print(premium.__balance)
except AttributeError as error:
    print(f"AttributeError: {error}")

Reward of 100 added safely.
Premium balance: 2100
AttributeError: 'PremiumAccount' object has no attribute '__balance'


## 9. Class invariants

An **invariant** is a rule that should always be true for a valid object.

For a game character:

- Health must stay between 0 and 100.
- Damage cannot increase health.
- Healing cannot increase health above 100.

Encapsulated methods protect those rules after every change.

In [12]:
class GameCharacter:
    def __init__(self, name):
        self.name = name
        self.__health = 100

    @property
    def health(self):
        return self.__health

    def take_damage(self, amount):
        if amount < 0:
            raise ValueError("Damage cannot be negative.")
        self.__health = max(0, self.__health - amount)

    def heal(self, amount):
        if amount < 0:
            raise ValueError("Healing cannot be negative.")
        self.__health = min(100, self.__health + amount)


hero = GameCharacter("Nova")
hero.take_damage(35)
print("After damage:", hero.health)

hero.heal(1000)
print("After healing:", hero.health)

After damage: 65
After healing: 100


## 10. Computed properties

A property does not have to return a stored value. It can calculate a value whenever it is requested.

The rectangle stores only width and height. Its `area` property is calculated from them, so it can never become stale.

In [13]:
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    @property
    def area(self):
        return self.width * self.height


rectangle = Rectangle(4, 5)
print("Area:", rectangle.area)

rectangle.width = 10
print("New area:", rectangle.area)

Area: 20
New area: 50


## Encapsulation versus abstraction

These concepts are related but different:

| Concept | Main question | Easy example |
|---|---|---|
| Encapsulation | How do we bundle and protect data and behavior? | Change balance through `deposit()` |
| Abstraction | What simple controls should users see? | Show a Withdraw button, hide banking machinery |

Encapsulation builds a safe capsule around an object's state. Abstraction chooses the small, useful interface shown to users.

## Common mistakes

1. Believing `_name` is inaccessible. It is only a protected-use convention.
2. Believing `__name` is secure or encrypted. It is only name-mangled.
3. Writing getters and setters that perform no useful validation or transformation.
4. Returning a mutable internal list and letting callers bypass class rules.
5. Validating only during construction but not during later updates.
6. Changing private data through its mangled name from outside the class.
7. Hiding everything and making the class difficult to use or test.
8. Printing an error when callers need to handle it; reusable classes often benefit from raising clear exceptions.

## Good design rules

- Keep the public interface small and predictable.
- Protect important invariants inside the class.
- Prefer properties when attribute-style access is natural.
- Use methods for actions such as deposits, withdrawals, and adding members.
- Return copies or immutable views of sensitive collections.
- Use double underscores mainly to prevent accidental name clashes or access—not as a security feature.

## Easy revision cheat sheet

| Code | Meaning | Outside access? |
|---|---|---|
| `self.name` | Public attribute | Yes, intended |
| `self._name` | Protected convention | Possible, but use carefully |
| `self.__name` | Name-mangled attribute | `obj.__name` fails |
| `get_value()` | Traditional getter method | Yes |
| `set_value(x)` | Traditional setter method | Yes, with checks |
| `@property` | Attribute-style getter | Yes |
| `@value.setter` | Attribute-style validated setter | Yes |
| Property without setter | Read-only public view | Read yes, assign no |
| `tuple(self.__items)` | Immutable view of internal items | Cannot append to it |

### Five questions for quick revision

1. Why is `account.deposit(100)` safer than directly changing the balance?
2. What is the difference between `_name` and `__name`?
3. Why is name mangling not real security?
4. When is a property better than explicit getter and setter methods?
5. How can a class safely expose an internal list?

### Five-second revision

**Encapsulation keeps data and behavior together and protects important rules by making callers use a clear, safe interface.**